# NBA Player Points Forecasting — Complete Team Methods Walkthrough

**Protocol: `blueprint_v1` · Status: validation-stage RQ1/RQ2 completed; RQ3 pending**

This notebook explains the complete implemented workflow: **research question → NBA extraction → validation and cleaning → pregame feature engineering → chronological splits → model fitting and tuning → evaluation → candidate lock → export and inference**. Each stage states its inputs, processing rules, outputs, and rationale, with executable intermediate tables.

The project builds **one general model across eligible NBA players**. Scottie Barnes is used below to make the transformations readable; he is an example, not the project's entire modeling scope. Each experiment row is one player's appearance in one game, and the target is that game's points, `target_points`.

The bundled preparation walkthrough uses actual 2024–25 records. The full experiment uses league-wide training and validation exports. These two datasets have separate purposes. No 2026–27 final-test file is downloaded, read, or scored by this notebook.

Earlier exploratory experiments used an additional 2020–21 training season and HistGradientBoosting. Those experiments are outside this focused PR; their results remain separate from the official protocol.

## 1. Prediction task and research questions

At prediction time, the player, game date, team and opponent are assumed known. We estimate **the player's total points in that game**, conditional on the player appearing. We do not predict participation, in-game events, or the next ten minutes of play. A “last-ten” feature means ten previous **player appearances**, not ten minutes or ten team games.

| Research question | Comparison | Required evidence |
| --- | --- | --- |
| RQ1: Which algorithm has the lowest scoring error? | M0 historical mean, M1 Ridge, M2 Random Forest, M3 XGBoost | Four-model MAE/RMSE table and actual-versus-predicted plots |
| RQ2: Which feature categories help? | F1 history, F2 + opponent, F3 + game context, for each ML model | Nine matched-cohort results and incremental MAE changes |
| RQ3: Does the locked procedure generalize? | Genuinely unseen 2026–27 games | Reserved final-test results and error distributions; still pending |

Using all players gives the model many examples of how recent scoring, playing time, shot volume, and game context relate to points. Player names and IDs are used to build histories and report predictions; they are not model inputs. A player appearing in both training and validation is consistent with forecasting future games for known players. This is not a held-out-player generalization experiment.

## 2. Setup, file locations, and execution modes

From the project root, run `uv sync --frozen --group dev`, then select the project's `.venv` Python kernel in VS Code/Jupyter. The full downloadable package includes the prepared CSVs, results, frozen model, and small raw-data walkthrough.

Run All performs the preparation walkthrough, audits the official split files, reads verified saved experiment results, reproduces the selected training-only fit, and demonstrates target-free inference. If the result manifest is absent, `RUN_EXPERIMENT` below becomes true and the fixed 39-candidate search runs. Choose a fresh results directory for an intentional new run.

The source notebook is output-free for Git review. The executed copy contains the intermediate tables, explanations and figures for reading without rerunning. This notebook imports shared repository functions so the educational and batch workflows use the same transformation and modeling rules.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts" / "experiment_config.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the 4451-Final-Project checkout.")
sys.path.insert(0, str(REPO_ROOT))

import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Code
from scripts.experiment_config import SPLITS, SPLIT_FILES, FEATURE_GROUPS, FIXED, CANDIDATES, protocol_spec, protocol_sha256
from scripts.blueprint import run_blueprint, prepare_blueprint, build_estimator, fit_matrix
from scripts.features import (FEATURE_COLUMNS, add_player_history, add_team_schedule,
                             add_team_ratings, pair_opponents, build_features)
from scripts.prepare_data import separate_ready_rows
from scripts.modeling import (METADATA_COLUMNS, TARGET, file_sha256, load_clean_split,
                              preparation_report, validate_split_order, regression_metrics)
from scripts.model_artifact import load_artifact, predict_frame, runtime_versions
from scripts.blueprint_report import read_verified_results, render_figures

CLEAN_DIR = REPO_ROOT / "data" / "clean" / "blueprint_v1"
RESULTS_DIR = REPO_ROOT / "artifacts" / "blueprint_v1" / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
WALKTHROUGH_DIR = REPO_ROOT / "examples" / "blueprint_walkthrough"
MODEL_DIR = REPO_ROOT / "artifacts" / "blueprint_v1" / "frozen_model"
# Run once when results are absent. Existing results are verified and reused.
RUN_EXPERIMENT = not (RESULTS_DIR / "experiment_manifest.json").is_file()
CANDIDATE_GRIDS = None  # None = the full, fixed official grid. Do not alter after observing results.

print("Data:", CLEAN_DIR.resolve())
print("Results:", RESULTS_DIR.resolve())
print("Protocol SHA-256:", protocol_sha256())

### 2.1 What is reproduced, and when?

| Stage | Default behavior | Network/model fitting |
| --- | --- | --- |
| Collection | Disabled; use the supplied prepared files | NBA requests only if explicitly enabled below |
| Small preparation example | Rebuild features from the bundled actual NBA subset | Offline; no model fitting |
| Full experiment | Verify/reuse results when present; otherwise run the declared grid | Training-only fitting when a fresh run is requested |
| Selected-model reproduction | Fit the already selected parameters once and compare saved validation predictions | Training only; no search or validation refit |
| Inference | Load the frozen artifact when available, or use the reproduced estimator | No additional fitting or NBA requests |

The data path must contain **engineered clean features**, not the original `player_game_logs_*.csv` files. The official train/validation CSVs each have 8 metadata columns, 18 numeric input columns, and 1 target column: 27 columns in total.

## 3. NBA extraction and source provenance

The batch collector is `scripts.nba_source.NBAData`. It requests regular-season records through `nba_api`, validates required source columns, and caches CSVs under `data/dirty`.

| Source | Endpoint / request setting | Fields used by this project | Cached location |
| --- | --- | --- | --- |
| Player game logs | `LeagueGameLog`, player mode, `Regular Season` | IDs, names, team/game/date, PTS, MIN, FGA, FG3A, FTA | `<season>/players.csv` |
| Team game logs | `LeagueGameLog`, team mode, `Regular Season` | Team/game/date and MATCHUP | `<season>/teams.csv` |
| Player profiles | `CommonPlayerInfo` | PERSON_ID and BIRTHDATE only | `profiles/<player_id>.csv` |
| Historical team summaries | `LeagueDashTeamStats`, Advanced, DateTo = target date minus one day | GP, DEF_RATING, PACE | `<season>/team_stats/<cutoff>.csv` |

Other fields may be present in the source files, but the input allowlist below determines what reaches a model. In particular, current-game shooting percentages, win/loss and plus/minus are not predictors. Historical height, roster, injury, and lineup features are not implemented here.

The first collection can be slow because profiles and daily advanced snapshots require many requests. Existing cache files are reused; downloads use finite retries and timeouts, and completed CSV writes replace temporary files atomically. Offline mode raises an error for a missing cache file rather than inventing a substitute.

The source game logs are expected to describe completed appearances. This implementation validates the required schema and historical values; it does not implement a separate live game-status feed. Completed-game availability and the test cutoff must be checked when collecting the ongoing test season.

In [ ]:
# Optional collection entrypoint. Leave disabled when using the supplied package.
RUN_COLLECTION = False
COLLECTION_OFFLINE = False  # True requires the complete original NBA cache.
if RUN_COLLECTION:
    prepare_blueprint(REPO_ROOT / "data", CLEAN_DIR, offline=COLLECTION_OFFLINE)
else:
    print("Collection skipped: using bundled walkthrough data and existing official exports.")

The official command-line equivalent is:

```sh
uv run --frozen python -m scripts.blueprint prepare
```

This calls the existing `prepare_season()` routine for the four training seasons and one validation season, then writes official combined files and reports under `data/clean/blueprint_v1`. It does not prepare the reserved test. Preparation refuses to overwrite a nonempty official output directory; choose a fresh `--output-dir` for another collection.

### 3.1 A bundled actual-data walkthrough

To make every transformation inspectable without another NBA download, `examples/blueprint_walkthrough` contains Scottie Barnes's first 14 appearances in 2024–25, his birthdate, all league team games from season start through the last included appearance, and the corresponding prior-day advanced snapshots. The expected clean output is an exact subset of the official league-wide prepared season.

Keeping the full team schedule matters: if Barnes misses games, Toronto's rest is still based on Toronto's previous game, not Barnes's previous appearance. The sample is selected chronologically, not by prediction accuracy. It is never substituted for the league-wide training or validation cohort.

In [ ]:
walkthrough_manifest = json.loads((WALKTHROUGH_DIR / "walkthrough_manifest.json").read_text())
for name, record in walkthrough_manifest["files"].items():
    assert file_sha256(WALKTHROUGH_DIR / name) == record["sha256"], name

raw_players = pd.read_csv(WALKTHROUGH_DIR / "player_game_logs.csv", dtype={"GAME_ID": "string"})
raw_teams = pd.read_csv(WALKTHROUGH_DIR / "team_game_logs.csv", dtype={"GAME_ID": "string"})
birthdates = pd.read_csv(WALKTHROUGH_DIR / "birthdates.csv")
snapshots = pd.read_csv(WALKTHROUGH_DIR / "team_snapshots.csv")
raw_players["GAME_DATE"] = pd.to_datetime(raw_players["GAME_DATE"])
raw_teams["GAME_DATE"] = pd.to_datetime(raw_teams["GAME_DATE"])
raw_players = raw_players.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)

display(pd.DataFrame([{"file": name, "rows": item["rows"], "hash_verified": True}
                      for name, item in walkthrough_manifest["files"].items()]))
display(raw_players[["PLAYER_NAME", "GAME_ID", "GAME_DATE", "MATCHUP", "PTS", "MIN", "FGA"]].head(5))
print("Walkthrough range:", walkthrough_manifest["first_game_date"], "through", walkthrough_manifest["last_game_date"])
print("This example is separate from the league-wide experiment.")

## 4. Player-level validation and lagged histories

`add_player_history()` normalizes dates and requires IDs and a nonempty player name. Historical PTS, MIN, FGA, FG3A and FTA must be numeric, present and nonnegative. Duplicate season/game/player records are rejected. Player-game rows are sorted by player, season, game date, and game ID before calculations.

Histories are grouped by **player and season**. They reset at a new season, but a trade does not reset a player's history. A player absence contributes no appearance and is not inserted as a zero-point game. Birthdate joins use a many-to-one constraint, and continuous age is computed for the actual game date.

The raw PTS value becomes the target for its own row. Earlier PTS values can contribute to later rows' histories only after those earlier games have finished.

In [ ]:
history = add_player_history(raw_players, birthdates)
display(history[["GAME_DATE", "PTS", "MIN", "points_last_5", "points_last_10",
                 "points_season_avg", "minutes_last_10", "games_played_before", "age"]])
assert history["games_played_before"].tolist() == list(range(len(raw_players)))

### 4.1 Why `shift(1)` comes before `rolling()`

For a player's appearance $t$ within a season, the previous-$k$ scoring average is

$$\operatorname{points\_last\_k}_{t} = \frac{1}{k}\sum_{j=1}^{k}\operatorname{PTS}_{t-j}.$$

The current game's $\operatorname{PTS}_t$ is excluded. The implementation is `values.shift(1).rolling(k, min_periods=k).mean()`. Applying a rolling mean without the shift would include the quantity we are trying to predict.

The season-to-date average is

$$\operatorname{points\_season\_avg}_{t} = \frac{1}{t-1}\sum_{j=1}^{t-1}\operatorname{PTS}_j,$$

and is missing at the first appearance. A ten-appearance mean remains missing until the eleventh appearance. These missing values represent unavailable history, not zero performance. Identical lagging rules apply to minutes and shot-attempt counts.

The next cell reconstructs the first eligible ten-game average manually and compares it with the shared implementation.

In [ ]:
first_eligible = history.loc[history["games_played_before"].eq(10)].iloc[0]
prior_ten = raw_players.loc[raw_players["GAME_DATE"].lt(first_eligible["GAME_DATE"])].tail(10)
manual_mean = prior_ten["PTS"].mean()
np.testing.assert_allclose(manual_mean, first_eligible["points_last_10"])
display(prior_ten[["GAME_DATE", "PTS"]])
display(pd.DataFrame([{"target_game_date": first_eligible["GAME_DATE"],
    "actual_points_in_target_game": first_eligible["PTS"], "previous_ten_points_mean": manual_mean,
    "stored_points_last_10": first_eligible["points_last_10"], "prior_appearances": len(prior_ten)}]))

## 5. Team schedule, historical snapshots, and opponents

### 5.1 Rest and back-to-back calculations

`add_team_schedule()` requires exactly two distinct team rows per game, a shared game date, and distinct increasing dates within each team/season. `is_home` is derived from the team log's `MATCHUP` field.

If the previous team game was on November 1 and the target game is November 2, the gap is one calendar day and **full rest days = 0**. The formula is `date_gap_days - 1`; back-to-back is 1 when `date_gap_days == 1`. A team's first game has no prior rest interval and is left missing.

`expected_prior_games` counts all previous games for the team. It later verifies that an advanced summary did not accidentally include the target game or the full season.

In [ ]:
schedule = add_team_schedule(raw_teams)
known_rest = schedule["team_rest_days"].notna()
assert schedule.loc[known_rest, "team_back_to_back"].eq(
    schedule.loc[known_rest, "team_rest_days"].eq(0).astype(float)).all()
example_team = int(raw_players.iloc[0]["TEAM_ID"])
display(schedule.loc[schedule["TEAM_ID"].eq(example_team),
    ["GAME_DATE", "MATCHUP", "is_home", "team_rest_days", "team_back_to_back", "expected_prior_games"]].head(12))

### 5.2 Advanced statistics must stop before the predicted game

For a target game on date D, the collector requests a season summary with `DateTo = D - 1 day`. `DEF_RATING` measures points allowed per 100 possessions; `PACE` estimates possessions per 48 minutes. These are cumulative season-to-cutoff summaries, not statistics from the target game.

`add_team_ratings()` requires `STATS_THROUGH < GAME_DATE`, joins on season/date/team, and checks that snapshot `GP` equals the team's number of earlier games. If an API ignores its date filter and returns a full-season summary, the GP mismatch causes a failure. Teams with prior games require nonmissing pace and defensive rating. Teams with no prior games have unavailable ratings.

These checks enforce the implemented cutoff contract on retrospective NBA queries. They do not establish when a later statistical correction first became available in a live feed; that is a limitation of retrospective data.

In [ ]:
rated_teams = add_team_ratings(schedule, snapshots)
assert rated_teams["GP"].fillna(0).eq(rated_teams["expected_prior_games"]).all()
assert (pd.to_datetime(snapshots["STATS_THROUGH"]) < pd.to_datetime(snapshots["GAME_DATE"])).all()
display(rated_teams.loc[rated_teams["TEAM_ID"].eq(example_team),
    ["GAME_DATE", "expected_prior_games", "GP", "PACE", "DEF_RATING"]].head(12))

### 5.3 Attach the other team's context

`pair_opponents()` pairs the two team rows within the same season/game and keeps the other team's ID, defensive rating, pace, rest, and back-to-back status. The own-team pace is renamed `team_pace`.

Player rows join this context on **season, game ID, team ID, and game date** with a many-to-one constraint. Every player appearance must match a team-game context. This distinguishes a traded player's team for that particular game from a current roster field.

In [ ]:
context = pair_opponents(rated_teams)
player_context = raw_players[["season", "GAME_ID", "GAME_DATE", "TEAM_ID"]].merge(context,
    on=["season", "GAME_ID", "GAME_DATE", "TEAM_ID"], validate="one_to_one")
display(player_context[["GAME_DATE", "TEAM_ID", "OPPONENT_ID", "is_home", "team_rest_days",
                        "opponent_rest_days", "team_pace", "opponent_pace", "opponent_defensive_rating"]])

## 6. Assemble the model table and explain exclusions

`build_features()` performs the shared player-history/context join, renames current PTS to `target_points`, and returns only the metadata, explicit 18-feature allowlist and target. Metadata names become lowercase to form the clean schema.

`separate_ready_rows()` rejects infinite values, retains rows with no missing input/target values, and returns an exclusion table with `missing_columns` for each omitted appearance. It does not fill missing histories with zeros or fit an imputer. The eleventh appearance is the first that can have all required ten-appearance histories.

The next cell executes these functions on the actual subset and checks the result against four reference rows extracted from the official full-league export. The expected 14 → 4 retained + 10 excluded transition is an example of the production rule, not a sample used to train the models.

In [ ]:
rebuilt = build_features(raw_players, raw_teams, birthdates, snapshots)
demo_clean, demo_excluded = separate_ready_rows(rebuilt)
demo_clean = demo_clean.sort_values(["game_date", "game_id", "player_id"]).reset_index(drop=True)
expected_demo = pd.read_csv(WALKTHROUGH_DIR / "expected_clean_rows.csv", dtype={"game_id": "string"})
expected_demo["game_date"] = pd.to_datetime(expected_demo["game_date"])
pd.testing.assert_frame_equal(demo_clean, expected_demo, check_dtype=False, rtol=1e-10, atol=1e-10)
assert len(rebuilt) == len(demo_clean) + len(demo_excluded)
display(pd.DataFrame({"stage": ["Raw player appearances", "Complete clean rows", "Excluded rows"],
                      "rows": [len(raw_players), len(demo_clean), len(demo_excluded)]}))
display(demo_excluded[["game_date", "games_played_before", "missing_columns"]])
display(demo_clean[["game_date", "points_last_10", "minutes_last_10", "opponent_defensive_rating", TARGET]])
print("Rebuilt rows match the official season-export subset.")

## 7. Demonstrate the leakage boundary

A useful negative control changes current and future outcomes in a temporary copy. Features available before the first eligible game must remain identical even when that game's points, minutes and attempts change. The target for that game should change, demonstrating that the test really altered the outcome.

The deliberately altered values below are a test perturbation in memory. They are never saved to the source files or used for training, tuning or reported NBA accuracy.

In [ ]:
cutoff = pd.Timestamp(first_eligible["GAME_DATE"])
perturbed_players = raw_players.copy()
perturbed_players.loc[perturbed_players["GAME_DATE"].ge(cutoff), ["PTS", "MIN", "FGA", "FG3A", "FTA"]] += 999
perturbed_features = build_features(perturbed_players, raw_teams, birthdates, snapshots)
keys = ["game_date", "game_id", "player_id"]
before_cutoff = rebuilt.loc[rebuilt["game_date"].le(cutoff)].sort_values(keys).reset_index(drop=True)
after_cutoff = perturbed_features.loc[perturbed_features["game_date"].le(cutoff)].sort_values(keys).reset_index(drop=True)
pd.testing.assert_frame_equal(before_cutoff[FEATURE_COLUMNS], after_cutoff[FEATURE_COLUMNS])
original_target = rebuilt.loc[rebuilt["game_date"].eq(cutoff), TARGET].iloc[0]
changed_target = perturbed_features.loc[perturbed_features["game_date"].eq(cutoff), TARGET].iloc[0]
assert changed_target == original_target + 999
print("Passed: current/future outcome changes do not alter earlier or current pregame inputs.")

invalid_snapshots = snapshots.copy()
invalid_snapshots["STATS_THROUGH"] = invalid_snapshots["GAME_DATE"]
try:
    add_team_ratings(schedule, invalid_snapshots)
except ValueError as error:
    print("Expected cutoff rejection:", error)
else:
    raise AssertionError("A same-day snapshot must be rejected.")

## 8. Full feature dictionary and nested categories

The table below gives every model input's meaning and first feature group. F2 contains all of F1; F3 contains all of F2. Their column **membership** is nested even though each group's input order is separately fixed. Age and prior-appearance count are common controls from F1 onward.

In [ ]:
feature_definitions = {
    "points_last_5": ("Mean PTS in the previous 5 appearances", "points"),
    "points_last_10": ("Mean PTS in the previous 10 appearances", "points"),
    "points_season_avg": ("Mean PTS across every earlier appearance in this season", "points"),
    "minutes_last_5": ("Mean MIN in the previous 5 appearances", "minutes"),
    "minutes_last_10": ("Mean MIN in the previous 10 appearances", "minutes"),
    "fga_last_10": ("Mean field-goal attempts in the previous 10 appearances", "attempts/game"),
    "fg3a_last_10": ("Mean three-point attempts in the previous 10 appearances", "attempts/game"),
    "fta_last_10": ("Mean free-throw attempts in the previous 10 appearances", "attempts/game"),
    "age": ("(Game date - birthdate) in days / 365.2425", "years"),
    "games_played_before": ("Earlier appearances by this player in this season", "appearances"),
    "opponent_defensive_rating": ("Opponent season-to-cutoff points allowed per 100 possessions", "points/100 possessions"),
    "opponent_pace": ("Opponent season-to-cutoff possessions per 48 minutes", "possessions/48 minutes"),
    "team_pace": ("Own team's season-to-cutoff possessions per 48 minutes", "possessions/48 minutes"),
    "is_home": ("1 at home; 0 away, using the team-game matchup", "binary"),
    "team_rest_days": ("Full calendar days since own team's previous game", "days"),
    "opponent_rest_days": ("Full calendar days since opponent's previous game", "days"),
    "team_back_to_back": ("1 if own team also played on the preceding date", "binary"),
    "opponent_back_to_back": ("1 if opponent also played on the preceding date", "binary"),
}
assert set(feature_definitions) == set(FEATURE_COLUMNS)
display(pd.DataFrame([{"feature": name, "first_group": next(group for group, cols in FEATURE_GROUPS.items() if name in cols),
                       "definition": feature_definitions[name][0], "unit": feature_definitions[name][1]}
                      for name in FEATURE_COLUMNS]))
assert all(TARGET not in cols and "player_id" not in cols and "MIN" not in cols for cols in FEATURE_GROUPS.values())

## 9. Chronological split design and complete-case coverage

Training uses 2021–22 through 2024–25; validation uses 2025–26; the reserved test is 2026–27. This resolves “2021–25 Train” into exact season labels. Randomly splitting rows could place later games in training and earlier games in validation, so the split is chronological and entire games/seasons remain disjoint.

All eleven declared variants—the two M0 rules and nine ML combinations—use the same upstream full-18-feature complete-case cohort. We do not recover extra rows for F1 when an opponent/context value is missing. This keeps the ablation comparison on the same player-games, while restricting the population to that complete-case cohort.

`load_clean_split()` validates schema, string game IDs, dates, seasons, keys, numeric completeness, nonnegative values, binary flags and integer prior-appearance counts of at least ten. `preparation_report()` reconciles source = clean + excluded for each season. Missing-feature counts can overlap; adding them is not the number of excluded rows.

In [ ]:
train = load_clean_split(CLEAN_DIR / SPLIT_FILES["train"], SPLITS["train"])
validation = load_clean_split(CLEAN_DIR / SPLIT_FILES["validation"], SPLITS["validation"])
validate_split_order(train, validation)
coverage_parts = []
for split, frame in [("train", train), ("validation", validation)]:
    report, coverage, _ = preparation_report(CLEAN_DIR, split, frame, SPLITS[split])
    if report is None:
        raise ValueError("Official preparation reports must accompany the CSVs.")
    coverage_parts.append(coverage)
display(pd.concat(coverage_parts, ignore_index=True))
display(pd.DataFrame([{"split": label, "rows": len(frame), "eligible_players": frame["player_id"].nunique(),
                       "first_game": frame["game_date"].min(), "last_game": frame["game_date"].max()}
                      for label, frame in [("train", train), ("validation", validation)]]))
print("Training target summary:")
display(train[TARGET].describe().to_frame())
print("Model input shapes:", {group: (len(train), len(features)) for group, features in FEATURE_GROUPS.items()})

## 10. Model definitions and why each is included

### M0 — short-term and season-to-date historical means

M0 is a non-ML reference: $\hat y_t$ is either `points_last_10` or `points_season_avg`. Neither rule estimates coefficients from training labels. Both use only the player's earlier appearances. The ten-game rule reacts more quickly to recent changes; the season rule uses a longer history and can react more slowly. Both baselines are scored on the same eligible validation rows, and validation MAE selects the rule shown in the main RQ1 table.

### M1 — StandardScaler followed by Ridge regression

For feature $j$, the scaler learns the training mean $\mu_j$ and standard deviation $s_j$ and applies $z_{ij}=(x_{ij}-\mu_j)/s_j$ to training and validation. Validation is transformed using those same training statistics. The target remains in points.

Ridge estimates an intercept and coefficients by minimizing

$$\sum_{i\in\mathrm{train}}(y_i-b-\mathbf z_i^\top\boldsymbol\beta)^2 + \alpha\sum_j\beta_j^2.$$

The L2 penalty shrinks coefficients and is useful when rolling averages are correlated. The pipeline keeps scaling inside the fitted estimator, so export and inference apply the same transformation. We examine five alpha values and choose by validation MAE. The intercept is not penalized. Ridge provides a linear reference; its coefficients describe conditional associations, not causal effects.

### M2 — Random Forest regression

A forest averages the predictions of $B=150$ decision trees:

$$\hat y(\mathbf x)=\frac{1}{B}\sum_{b=1}^{B}T_b(\mathbf x).$$

Each tree uses a bootstrap sample of the **training rows only**. Trees can learn nonlinear thresholds and interactions, such as a different relationship between recent minutes and points at different shot volumes. Here all features are considered at a split (`max_features=1.0`); randomness comes from bootstrap samples and the seeded tree construction. Four depth/minimum-leaf combinations limit complexity. No feature scaling is applied.

Bootstrapping training rows is an algorithmic operation inside the training period. It does not change the chronological training/validation boundary. Random Forest is included to test whether nonlinear partitions improve on the linear model.

### M3 — actual XGBoost regression

XGBoost builds an additive ensemble of trees sequentially:

$$f_m(\mathbf x)=f_{m-1}(\mathbf x)+\eta h_m(\mathbf x).$$

Each new tree uses loss-gradient information to improve the current ensemble, with regularization controlling complexity. This protocol uses `XGBRegressor` from the `xgboost-cpu` distribution, 300 trees, learning rate 0.05, squared-error training loss, CPU histogram splits, and no early stopping. Four depth/L2 combinations are compared. Row and column subsampling are both 1.0. No input scaling is applied.

The training objective is squared error for Ridge, Random Forest and XGBoost; **validation selection uses MAE**. Those are different roles. M3 is not the HistGradientBoosting implementation used in earlier exploratory notebooks.

The next cell shows the actual factory and declared parameters used by the batch experiment, rather than a separately reimplemented training path.

In [ ]:
display(Code(inspect.getsource(build_estimator), language="python"))
display(pd.DataFrame([{"model_id": model_id, "fixed_parameters": json.dumps(FIXED[model_id], sort_keys=True),
                       "candidates": json.dumps(CANDIDATES[model_id], sort_keys=True)}
                      for model_id in ["M1", "M2", "M3"]]))
for model_id in ["M1", "M2", "M3"]:
    print(model_id, "constructor preview:", build_estimator(model_id, CANDIDATES[model_id][0]))

## 11. Fixed search, validation selection, and fair comparisons

For each ML family and feature group, the engine executes this sequence:

1. Select only that group's named feature columns and the training target.
2. Construct one estimator for each declared candidate configuration.
3. Fit on training rows; predict the later validation rows.
4. Compute validation MAE/RMSE and record every candidate score.
5. Keep the candidate with the lowest validation MAE; exact ties retain grid order.

There are `3 × (5 + 4 + 4) = 39` candidate fits across F1/F2/F3. Each family/group retains its own best configuration. Seeds, thread limits, grids and iteration counts are recorded in the protocol. No candidate is fitted on validation, and XGBoost does not use validation for early stopping.

The main RQ1 comparison fixes the feature group to F3 for M1/M2/M3 and includes the selected M0 rule. RQ2 keeps all nine ML/group combinations. Overall candidate lock then considers both M0 rules and all nine ML variants. Exact overall ties prefer fewer required inputs, then model/group/variant order.

This is a **validation development experiment**, not nested cross-validation. Reusing validation to tune and compare models can make the selected score optimistic; the reserved final test is needed for the final generalization claim. The declared grid has not been changed in response to the observed ranking.

In [ ]:
display(pd.DataFrame([{"split": split, "seasons": ", ".join(seasons)} for split, seasons in SPLITS.items()]))
display(pd.DataFrame([{"feature_set": group, "input_count": len(features), "columns": ", ".join(features)}
                      for group, features in FEATURE_GROUPS.items()]))
print("Declared candidate fits:", len(FEATURE_GROUPS) * sum(len(grid) for grid in CANDIDATES.values()))

## 12. Execute or verify the official experiment

`run_blueprint()` loads the two official clean splits, verifies preparation reports and chronological separation, executes the fixed search, and writes tables, predictions, coverage and an experiment manifest. The manifest records input/report hashes, exact configuration, selected parameters, runtime versions and code hashes.

When saved results exist, `read_verified_results()` checks the current data/report hashes, confirms that saved prediction keys/dates/targets match validation, and recomputes every variant's metrics from the saved predictions. Reading verified results avoids rerunning a search merely to recreate figures. A missing or mismatched input causes a failure.

The cell reports which mode it used. Its final-test flags must remain false. The only active experiment inputs are training and validation.

In [ ]:
print("Execution mode:", "fresh fixed-grid search" if RUN_EXPERIMENT else "verify saved experiment results")
if RUN_EXPERIMENT:
    fitted = run_blueprint(CLEAN_DIR, RESULTS_DIR, candidate_grids=CANDIDATE_GRIDS)
    del fitted  # Saved evidence is the interface for subsequent report cells.
tables = read_verified_results(CLEAN_DIR, RESULTS_DIR)
manifest = tables["manifest"]
assert not manifest["final_test_loaded"] and not manifest["final_test_evaluated"]
print("Full official grid:", manifest["is_full_protocol_grid"])
print("RQ3:", manifest["rq3_status"])
display(tables["preparation_coverage"])

## 13. Evaluation metrics and per-game errors

For $n$ eligible validation appearances, observed points $y_i$ and predictions $\hat y_i$:

$$\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|,$$

$$\operatorname{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2}.$$

Both are measured in points. MAE is the primary selection metric and gives each appearance equal weight. RMSE penalizes larger misses more strongly, so two candidates can rank differently under the two metrics. An MAE of 4.66 describes average absolute error in this cohort; it is not a guarantee that every prediction is within 4.66 points.

We retain fractional predictions and apply no clipping or rounding. Current outcomes are used here only to measure error after prediction. The next cell independently checks one baseline's metrics and exposes the first few chronological prediction errors.

In [ ]:
predictions = tables["predictions"]
baseline_variant = "M0_previous_10"
error = predictions[baseline_variant].to_numpy(dtype=float) - predictions[TARGET].to_numpy(dtype=float)
manual_metrics = {"n": len(error), "MAE": float(np.mean(np.abs(error))), "RMSE": float(np.sqrt(np.mean(error ** 2)))}
recorded = tables["baseline_reference"].set_index("variant").loc[baseline_variant]
np.testing.assert_allclose([manual_metrics["MAE"], manual_metrics["RMSE"]], [recorded["MAE"], recorded["RMSE"]], rtol=1e-10)
print("Independently reconstructed baseline metrics:", manual_metrics)
error_example = predictions[["game_date", "player_name", TARGET, baseline_variant]].head(5).copy()
error_example["signed_error"] = error_example[baseline_variant] - error_example[TARGET]
error_example["absolute_error"] = error_example["signed_error"].abs()
display(error_example)

## 14. RQ1 — official four-model validation results

The first table retains both M0 rules. The main table has exactly four rows: the validation-selected historical average and M1/M2/M3 with all F3 inputs. All models score the same validation player-games, so the error differences are not caused by different sample sizes.

The bar plots show absolute error levels. The four density scatter panels compare actual and predicted points; darker bins contain more appearances. The dashed identity line marks perfect predictions. Points above that line are overpredictions, and points below it are underpredictions.

Read the numerical differences at their actual scale. In the verified official run, the F3 Ridge and XGBoost MAEs differ by about 0.001 point. This experiment does not show that such a small difference is statistically reliable or practically meaningful.

In [ ]:
display(tables["baseline_reference"])
display(tables["rq1_model_comparison"][["model_id", "variant", "feature_set", "n", "MAE", "RMSE"]])
figures = render_figures(tables, FIGURE_DIR)
display(figures[0])
display(figures[1])
plt.close("all")

### 14.1 Interpret the comparison without changing the protocol

A historical average is a strong reference because recent player scoring already captures some role, playing-time and ability information. A trained model can improve by combining related historical signals, but a nonlinear algorithm is not guaranteed to win. Training objective, sample size, regularization and the declared parameter budget can all affect the ranking.

The official run's F3 comparison favors Ridge by observed validation MAE. These results do not prove Ridge is universally the best NBA algorithm. They compare the declared fitted procedures under this split and parameter budget. Do not insert the older five-season HistGradientBoosting score into this table: both its training period and method differ.

## 15. RQ2 — feature contribution under matched cohorts

For each of M1/M2/M3, compare F1 → F2 → F3. Define the incremental improvement as:

`previous_feature_set_MAE - current_feature_set_MAE`.

A positive number means error decreased after adding the next category; a negative number means it increased. F1 has no preceding group. Every group uses the identical validation appearances, and each model/group is independently tuned under the same family-specific grid.

The increments therefore compare tuned procedures. They do not isolate a causal effect or a fixed-hyperparameter effect of the added fields. Correlated predictors, redundant information and limited signal can make extra context unhelpful. F2 can worsen MAE while a later F3 addition partially recovers it; assess the full sequence rather than assuming more inputs must improve performance.

In [ ]:
display(tables["rq2_feature_ablation"][["model_id", "feature_set", "feature_count", "n", "MAE", "RMSE",
                                           "previous_feature_set", "incremental_MAE_reduction"]])
display(figures[2])
plt.close("all")

In the verified official run, F1 Ridge has the lowest overall MAE. Opponent context does not consistently reduce MAE, and F3 recovers part of the F2 increase for all three families. This supports a restrained statement: **the added context did not produce a consistent MAE improvement under this protocol**. It does not imply opponent defense or game context never matters.

This is a nested category ablation. It is not a permutation-importance study, an individual-feature ranking, or a significance test.

## 16. Candidate lock across the declared variants

The overall candidate is selected from both M0 rules plus all nine ML/group variants using validation MAE and the declared tie rule. It can differ from the winner of the F3-only RQ1 comparison. The official run selected **M1_F1**: scaled Ridge with 10 inputs and alpha 100, with validation MAE approximately 4.6573 and RMSE 6.0496 points.

That improves MAE over the selected ten-game mean by approximately 0.0589 points, or 1.25%, on these validation rows. The gain is modest and should be reported at that scale. The next cell derives the values from saved evidence rather than hard-coding a second set of results. For synthetic test fixtures, these displayed values will differ.

In [ ]:
selected = manifest["selected_candidate"]
display(pd.DataFrame([selected]))
selected_variant = selected["variant"]
print("Selected parameters:", manifest["selected_parameters"].get(selected_variant, "Non-ML historical mean rule"))
print("Tables and figures:", RESULTS_DIR.resolve())
baseline_mae = tables["baseline_reference"]["MAE"].min()
gain = baseline_mae - selected["MAE"]
print(f"MAE reduction versus selected M0: {gain:.6f} points ({100 * gain / baseline_mae:.2f}%)")

### 16.1 Reproduce the selected fit explicitly

This cell exposes the final fit/predict mechanics in the notebook. It constructs the already selected configuration, fits **training rows only**, predicts validation and checks against the frozen prediction column. It performs no hyperparameter search and does not use validation labels in `.fit()`.

For Ridge, the scaler's learned means are checked against the training feature means, demonstrating where preprocessing parameters came from. All feature selection is by the named allowlist. The held-out test is not involved.

In [ ]:
from threadpoolctl import threadpool_limits

selected_model_id = selected["model_id"]
selected_group = selected["feature_set"]
selected_params = manifest["selected_parameters"].get(selected_variant, {})
if selected_model_id == "M0":
    from scripts.blueprint_artifact import HistoricalAverage, BASELINES
    required_features = [BASELINES[selected_variant], "games_played_before"]
    reproduced_model = HistoricalAverage(BASELINES[selected_variant])
else:
    required_features = FEATURE_GROUPS[selected_group]
    selected_grid_values = {key: selected_params[key] for key in CANDIDATES[selected_model_id][0]}
    reproduced_model = build_estimator(selected_model_id, selected_grid_values)
    with threadpool_limits(limits=2):
        reproduced_model.fit(train[required_features], train[TARGET])

with threadpool_limits(limits=2):
    reproduced_predictions = np.asarray(reproduced_model.predict(validation[required_features]), dtype=float)
np.testing.assert_allclose(reproduced_predictions, predictions[selected_variant], rtol=1e-10, atol=1e-10)
print("Selected-fit reproduction passed:", regression_metrics(validation[TARGET], reproduced_predictions))
if selected_model_id == "M1":
    learned_scaler = reproduced_model.named_steps["scale"]
    np.testing.assert_allclose(learned_scaler.mean_, train[required_features].mean())
    assert learned_scaler.n_samples_seen_ == len(train)
    display(pd.DataFrame({"feature": required_features, "training_mean": train[required_features].mean().to_numpy(),
                          "scaler_mean": learned_scaler.mean_, "scaler_scale": learned_scaler.scale_}))

## 17. Frozen export and target-free inference

The exporter `scripts.blueprint_artifact` verifies the official full-grid protocol, recomputes the recorded selection rule, checks source CSV/report hashes, fits the selected configuration on training only, and reproduces saved validation predictions. It then saves and reloads the model and records the maximum prediction difference. Parallel forest reductions may differ at the last floating-point bits; reload checks use a strict numerical tolerance and record the observed difference.

The artifact directory contains the model, `model_manifest.json`, and `verification.json`. Ridge/Random Forest use trusted joblib files; XGBoost uses native JSON; M0 uses a JSON rule. The loader checks the model hash, feature/parameter contract and required runtime versions. Hashes check file identity; load joblib artifacts only from trusted team sources.

From the repository root, export into a new directory with:

```sh
uv run --frozen python -m scripts.blueprint_artifact
```

No validation refit or test-driven selection occurs in this export. The current team package already includes the verified artifact, so another export is unnecessary for reading results or predicting.

In [ ]:
# Prefer the verified saved model when it belongs to this exact experiment.
if (MODEL_DIR / "model_manifest.json").is_file():
    inference_model, inference_contract = load_artifact(MODEL_DIR)
    if inference_contract["source_experiment_manifest_sha256"] != file_sha256(RESULTS_DIR / "experiment_manifest.json"):
        raise ValueError("The frozen model belongs to a different experiment. Set MODEL_DIR correctly.")
    assert inference_contract["candidate"] == selected_variant
    print("Loaded frozen candidate:", inference_contract["candidate"])
    display(pd.DataFrame([json.loads((MODEL_DIR / "verification.json").read_text())]))
else:
    # The explicit training-only reproduction can still demonstrate inference.
    # This in-memory contract is not a claim that an artifact has been exported.
    inference_model = reproduced_model
    inference_contract = {"artifact_schema_version": 2, "protocol_sha256": protocol_sha256(),
        "model_id": selected_model_id, "feature_set": selected_group, "candidate": selected_variant,
        "feature_columns": required_features, "target_column": TARGET, "selected_parameters": selected_params,
        "model_filename": "baseline.json" if selected_model_id == "M0" else "model.json" if selected_model_id == "M3" else "model.joblib"}
    print("No saved artifact found: demonstrating inference with the reproduced training-only estimator.")

### 17.1 What an inference row must contain

The inference function accepts **prepared pregame features** for the selected variant and optional metadata. It does not need `target_points`, raw current-game minutes or outcomes. It maps columns by name, restores the artifact's required order, validates values and history eligibility, and preserves input row order.

The first five chronological eligible validation rows below are a historical execution demonstration, not a live future-game forecast or an accuracy-selected sample. The target is deliberately omitted and headers are reversed. Features for a genuinely upcoming game would first need to be assembled from completed history and the known schedule using the same definitions.

In [ ]:
example_input = validation[METADATA_COLUMNS + required_features].head(5)
example_input = example_input[list(reversed(example_input.columns))]
assert TARGET not in example_input
example_output = predict_frame(example_input, inference_model, inference_contract)
np.testing.assert_allclose(example_output["predicted_points"], predictions[selected_variant].head(5), rtol=1e-10, atol=1e-10)
display(example_input)
display(example_output)
print("Target-free, reordered-column inference reproduced the saved predictions.")

The command-line interface uses the same loader and prediction function:

```sh
uv run --frozen python -m scripts.predict --model-dir artifacts/blueprint_v1/frozen_model --input artifacts/blueprint_v1/demo/validation_pregame_sample.csv --output artifacts/my_predictions.csv
```

The CSV output contains `input_row`, available metadata and `predicted_points`; it does not copy the outcome into predictions. Missing/duplicate inputs, nonnumeric or nonfinite values, negative features, invalid required binary flags and insufficient history are rejected. There is no silent imputation, rounding or clipping. The command protects its input and frozen artifact files from overwrite.

## 18. Reproducibility evidence and artifact map

The focused PR suite checks data/leakage boundaries, experiments, notebook execution and artifact contracts. The actual validation-stage handoff executed the official notebook, reproduced all 21,204 validation predictions after export/reload, and reproduced them in a separate CLI process with no target and reversed input headers. Maximum differences in that run were zero. Tests include synthetic leakage mutations and serialization/contract checks; synthetic fixture scores are not NBA research results.

The following code hashes identify the experiment-producing modules. A changed hash means results belong to an earlier source version and require review. Runtime versions are also printed so another teammate can select the locked environment.

In [ ]:
display(pd.DataFrame([{"module": name, "recorded_sha256": digest,
                       "matches_current_source": file_sha256(REPO_ROOT / "scripts" / name) == digest}
                      for name, digest in manifest["code_sha256"].items()]))
display(pd.DataFrame([{"package": key, "experiment_version": value, "current_version": runtime_versions().get(key)}
                      for key, value in manifest["runtime_versions"].items()]))

| Artifact | Contents and purpose |
| --- | --- |
| `train_2021_2025.csv`, `validation_2025_2026.csv` | Official chronological engineered inputs and targets |
| `report_train.json`, `report_validation.json` | Per-season source, clean and excluded counts |
| `excluded/excluded_<season>.csv` | Omitted appearances and their missing input names |
| `rq1_model_comparison.csv` | Four official model results on the declared RQ1 setup |
| `baseline_reference.csv` | Both non-ML historical mean rules |
| `rq2_feature_ablation.csv` | Nine ML/group results and incremental MAE reductions |
| `candidate_scores.csv` | Every tested configuration's validation score |
| `validation_all_variants.csv` | Both M0 rules plus all nine ML variants |
| `validation_predictions.csv` | Same-row metadata, outcomes and all retained variants' predictions |
| `experiment_manifest.json` | Protocol, input/report hashes, versions, tuning choices, selection and test status |
| `figures/*.png` | Report-ready RQ1 metric/scatter and RQ2 ablation figures |
| Frozen model directory | Saved estimator/rule, manifest and reload verification |
| `examples/blueprint_walkthrough` | Small actual-data preparation example and reference clean output |

For a deliberate full rerun, choose fresh output paths and use:

```sh
uv run --frozen python -m scripts.blueprint evaluate --clean-dir data/clean/blueprint_v1 --output-dir artifacts/blueprint_v1/new_results
uv run --frozen python -m scripts.blueprint_artifact --clean-dir data/clean/blueprint_v1 --source-results-dir artifacts/blueprint_v1/new_results --output-dir artifacts/blueprint_v1/new_frozen_model
uv run --frozen --group dev python -m unittest discover -s tests -v
```

Keep code, lockfile, data/report hashes and results together. Do not replace historical evidence in place when changing a split, feature definition or search grid.

## 19. Limitations and interpretation boundaries

- **Appearance-conditioned prediction:** DNPs and availability are not modeled; the evaluation includes players who appeared.
- **Warm-up and complete-case restrictions:** First-ten-appearance and incomplete rows are excluded. Accuracy is not measured for all scheduled player-games or cold starts.
- **Within-season histories:** Previous-season form is not carried into rolling features; trades preserve same-season player history.
- **Information scope:** Injury reports, expected lineups, projected minutes, and within-game play-by-play are absent. Historical minutes are used, not actual minutes from the predicted game.
- **Retrospective sources:** Date filters and GP checks enforce pregame cutoffs on retrieved summaries, but later source corrections and historical publication timing are not fully reconstructed.
- **Validation selection:** Validation has been reused for parameter and candidate selection. No statistical significance or unseen-season reliability is established by these tables.
- **Parameter budget and correlated rows:** Conclusions concern these declared grids; repeated appearances by the same players and games create dependence. No independent-row significance test is claimed.

These limits should accompany the reported MAE/RMSE. Additional features or methods should become a separately versioned experiment with a predeclared comparison, rather than being selected after inspecting reserved test outcomes.

## 20. RQ3 plan — reserved, not executed

Before reading any 2026–27 outcome, record the frozen candidate, input group, parameters, training policy and test cutoff. This protocol's current artifact remains fitted on training only. Any different refit policy must be agreed and versioned before test access.

Prepare completed test appearances with the same pregame definitions and eligibility rules. When scoring a later game, histories may include earlier completed test-season games that would already have been available at prediction time. That is legitimate sequential information; using the current/future outcome is not. Do not update fitted model parameters during this locked evaluation.

Report final-test coverage, MAE/RMSE, error distributions and the exact date window. The ongoing season produces a partial-season conclusion. Do not retune or replace the candidate after reading test results. Earlier monthly validation analyses do not satisfy this held-out-season RQ3.

No executable test-loading or test-scoring cell is included here. This section documents the remaining milestone rather than presenting unperformed results.

## 21. Team milestones and IEEE report handoff

| Team milestone | Implemented evidence | Remaining work |
| --- | --- | --- |
| D1: collection, cleaning and leakage controls | Shared NBA pipeline, reports/exclusions, actual walkthrough, synthetic leakage tests | Team integration review; completed-game/test-cutoff audit for RQ3 |
| Phase 2: M0/M1 | Both historical means, scaled Ridge, common scoring framework | Final held-out evaluation |
| Phase 3: M2/M3 and RQ1 | Random Forest and actual XGBoost; four-model validation comparison | Final held-out evaluation |
| Phase 4: RQ2 | All nine nested-group combinations on identical rows | RQ3 still pending |
| D2/D3 | Reproducible validation pipeline, tables/figures/manifests, frozen export and inference | Final-test artifacts after candidate lock |
| D4: IEEE report | Methods walkthrough and report-ready validation evidence | Related work, complete Methods/Discussion with final results, limitations, conclusions and IEEE assembly |
| Git workflow | Local feature-branch source and reviewable patches | PR into `development`; stable promotion to `main` after team review |

Use Sections 1–13 for the Methods narrative, Sections 14–16 for validation Results/Discussion, and Sections 19–20 for limitations and remaining evaluation. The notebook is an implementation record, not a completed IEEE report or literature review.

## Appendix A. Source map and the executed search implementation

| Module | Responsibility |
| --- | --- |
| `scripts/nba_source.py` | Endpoint calls, required-column checks, cache, retries, atomic CSV writes |
| `scripts/features.py` | Lagged player histories, schedule/rest, snapshot cutoffs, opponent joins, input allowlist |
| `scripts/prepare_data.py` | Shared per-season transformation, complete-case split, exclusion/report writing; legacy CLI defaults remain separate |
| `scripts/experiment_config.py` | Official seasons, nested inputs, models, grids, rules and protocol hash |
| `scripts/blueprint.py` | Official preparation and train/validation experiment engine |
| `scripts/blueprint_report.py` | Saved-evidence verification and RQ1/RQ2 figures |
| `scripts/blueprint_artifact.py` | Generic official winner export/load contract |
| `scripts/model_artifact.py`, `scripts/predict.py` | Shared target-free inference and compatible loading |
| `tests/` | Synthetic preparation/leakage, notebook and artifact verification |

The exact current search function is displayed below so a teammate can inspect the fitting loops, candidate tie rule, same-row predictions, RQ1/RQ2 table construction and overall selection without leaving the notebook.

In [ ]:
display(Code(inspect.getsource(fit_matrix), language="python"))

## Appendix B. Common execution problems and implementation references

| Symptom | Check |
| --- | --- |
| Missing clean CSV | Confirm `CLEAN_DIR` points to the extracted `data/clean/blueprint_v1` directory; raw logs have a different schema |
| Missing walkthrough files | Use the updated full package, or copy its `examples/blueprint_walkthrough` directory into the checkout |
| Missing imports or artifact runtime mismatch | Run `uv sync --frozen --group dev` and select the project's `.venv` kernel |
| Existing output directory | Reuse verified results or choose a new results/export path for a deliberate rerun |
| Hash/report mismatch | Restore the matching data/reports/results; investigate rather than bypassing validation |
| Missing offline source cache | Use the supplied prepared files for modeling, or complete collection before enabling offline preparation |
| Frozen model belongs to another run | Set `MODEL_DIR` to the artifact whose source experiment hash matches `RESULTS_DIR` |

Implementation documentation: [nba_api](https://github.com/swar/nba_api), [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html), and [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html). These explain software behavior; they do not replace the academic related-work section or justify a final-test claim.